In [0]:
dbutils.widgets.text(
    "Environment",
    "dev",
    "Nom du catalogue"
)

dbutils.widgets.text(
    "RunType",
    "once",
    "once = batch, autre valeur = streaming"
)

dbutils.widgets.text(
    "ProcessingTime",
    "5 seconds",
    "Intervalle des micro-batchs"
)

### Paramètres d'exécution

Ces widgets permettent de contrôler le pipeline sans modifier le code.

- `Environment` : catalogue utilisé, ici `dev`.
- `RunType` : `once` pour traiter les données disponibles puis arrêter le pipeline.
- `ProcessingTime` : fréquence des micro-batchs en mode streaming.

Cela permettra plus tard de réutiliser le même notebook dans un Workflow Databricks.

In [0]:
env = dbutils.widgets.get("Environment")

once = (
    dbutils.widgets.get("RunType") == "once"
)

processing_time = dbutils.widgets.get(
    "ProcessingTime"
)

if once:
    print(
        f"Starting pipeline on {env} "
        f"in batch mode."
    )
else:
    print(
        f"Starting pipeline on {env} "
        f"in streaming mode "
        f"with {processing_time} micro-batches."
    )

### Lecture des paramètres

Cette partie récupère les valeurs des widgets et les transforme en variables Python.

Si `RunType = once`, le pipeline fonctionne comme un batch :

`nouvelles données → traitement → arrêt`

Sinon, il fonctionne en streaming continu.

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/02-setup

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/03-history-loader

In [0]:
SH = SetupHelper(env)

HL = HistoryLoader(env)

### Initialisation du projet

Cette partie crée les objets permettant d'utiliser les fonctions du Setup et du History Loader avec le catalogue choisi dans `Environment`.

In [0]:
setup_required = not spark.catalog.tableExists(
    f"{env}.bronze.registered_users_bz"
)

if setup_required:

    print("Setup required.")

    SH.setup()
    SH.validate()

else:

    print(
        "Project environment already exists. "
        "Skipping setup."
    )

### Vérification du Setup

Le notebook vérifie si l'environnement du projet existe déjà.

Si la table Bronze principale n'existe pas :

`Setup → création des schémas et tables`

Sinon, le Setup est ignoré afin d'éviter de recréer inutilement l'environnement.

In [0]:
HL.load_history()
HL.validate()

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/04-bronze

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/05-silver

In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/06-gold

In [0]:
BZ = Bronze(env)

SL = Silver(env)

GL = Gold(env)

### Initialisation des couches

Cette partie crée les trois objets permettant d'exécuter l'ensemble du pipeline.

`BZ → Bronze`

`SL → Silver`

`GL → Gold`

In [0]:
print("Starting Bronze...")

BZ.consume(
    once,
    processing_time
)

print("Bronze completed.")

### Exécution Bronze

La couche Bronze ingère les fichiers présents dans Azure vers les tables Delta Bronze.

`Landing Zone → Auto Loader → Bronze`

In [0]:
print("Starting Silver...")

SL.upsert(
    once,
    processing_time
)

print("Silver completed.")

### Exécution Silver

Une fois Bronze terminé, la couche Silver nettoie, transforme et enrichit les nouvelles données.

`Bronze → transformations → Silver`

In [0]:
print("Starting Gold...")

GL.upsert(
    once,
    processing_time
)

print("Gold completed.")